# JA→EN Translator – Exploration & Demo

This notebook walks through the complete pipeline:
1. **Tokeniser training** – build a shared SentencePiece vocabulary
2. **Data inspection** – visualise sequence-length distributions
3. **Model overview** – inspect architecture and parameter count
4. **Training demo** – run a small training loop on a toy dataset
5. **Translation demo** – translate sample sentences with a trained checkpoint

> **Prerequisites**: Install the dependencies first.
> ```bash
> pip install -r ../requirements.txt
> ```

In [ ]:
import sys, os
# Add the src/ directory to the Python path so we can import our modules
sys.path.insert(0, os.path.join('..', 'src'))

## 1 · Tokeniser Training

We use a **shared** SentencePiece BPE vocabulary for both Japanese and English.
The combined corpus is written to a temporary file and then passed to `train_tokenizer`.

In [ ]:
import tempfile, pathlib
from tokenizer import train_tokenizer, load_tokenizer, encode, decode

# --- Toy corpus (replace with your real data paths) ---
JA_SENTENCES = [
    "猫が窓の外を見ています。",
    "今日は良い天気ですね。",
    "私は毎日日本語を勉強しています。",
    "新幹線は世界で最も速い列車のひとつです。",
    "東京はとても大きな都市です。",
]
EN_SENTENCES = [
    "The cat is looking out the window.",
    "The weather is nice today.",
    "I study Japanese every day.",
    "The Shinkansen is one of the fastest trains in the world.",
    "Tokyo is a very large city.",
]

# Write combined corpus to a temp file
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as f:
    for line in JA_SENTENCES + EN_SENTENCES:
        f.write(line + '\n')
    corpus_path = f.name

model_prefix = '/tmp/toy_spm'
train_tokenizer(corpus_path, model_prefix, vocab_size=200, character_coverage=1.0)
print(f'Tokeniser model saved to {model_prefix}.model')

## 2 · Tokeniser Exploration

In [ ]:
sp = load_tokenizer(model_prefix + '.model')

for text in JA_SENTENCES[:3]:
    ids = encode(sp, text)
    pieces = sp.id_to_piece(ids)
    print(f'Input  : {text}')
    print(f'Pieces : {pieces}')
    print(f'IDs    : {ids}')
    print(f'Decoded: {decode(sp, ids)}')
    print()

## 3 · Data Inspection – Sequence Length Distribution

In [ ]:
import matplotlib.pyplot as plt

ja_lengths = [len(encode(sp, s)) for s in JA_SENTENCES]
en_lengths = [len(encode(sp, s)) for s in EN_SENTENCES]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(len(ja_lengths)), ja_lengths, color='steelblue')
axes[0].set_title('Japanese token lengths')
axes[0].set_xlabel('Sentence index')
axes[0].set_ylabel('Token count')

axes[1].bar(range(len(en_lengths)), en_lengths, color='coral')
axes[1].set_title('English token lengths')
axes[1].set_xlabel('Sentence index')
axes[1].set_ylabel('Token count')

plt.tight_layout()
plt.show()

## 4 · Model Architecture

In [ ]:
import torch
from model import Seq2SeqTransformer

vocab_size = sp.get_piece_size()

model = Seq2SeqTransformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=64,
    nhead=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    dim_feedforward=128,
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal trainable parameters: {total_params:,}')

## 5 · Mini Training Demo

A quick sanity-check: the loss should decrease over a handful of steps when overfitting on this tiny corpus.

In [ ]:
import torch.nn as nn
from torch.optim import Adam
from tokenizer import PAD_ID, BOS_ID, EOS_ID

device = torch.device('cpu')
model = model.to(device)
optimiser = Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def make_batch(ja_list, en_list):
    """Encode sentences and pad them into a single batch tensor."""
    from torch.nn.utils.rnn import pad_sequence
    src = pad_sequence(
        [torch.tensor(encode(sp, s), dtype=torch.long) for s in ja_list],
        batch_first=True, padding_value=PAD_ID)
    tgt = pad_sequence(
        [torch.tensor(encode(sp, s), dtype=torch.long) for s in en_list],
        batch_first=True, padding_value=PAD_ID)
    return src, tgt

src_batch, tgt_batch = make_batch(JA_SENTENCES, EN_SENTENCES)

losses = []
for step in range(50):
    tgt_input  = tgt_batch[:, :-1]
    tgt_labels = tgt_batch[:, 1:]

    logits = model(src_batch, tgt_input)
    loss   = criterion(logits.reshape(-1, logits.size(-1)), tgt_labels.reshape(-1))

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    losses.append(loss.item())

    if (step + 1) % 10 == 0:
        print(f'Step {step+1:>3}  loss={loss.item():.4f}')

plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Cross-entropy loss')
plt.title('Training loss (toy data – overfit check)')
plt.show()

## 6 · Translation Demo

After overfitting on the tiny corpus the model should roughly reconstruct the training targets.

In [ ]:
model.eval()

for ja in JA_SENTENCES:
    src_ids = encode(sp, ja, add_bos=True, add_eos=True)
    src_tensor = torch.tensor([src_ids], dtype=torch.long)
    tgt_ids = model.translate(src_tensor, bos_id=BOS_ID, eos_id=EOS_ID, max_len=50)
    translation = decode(sp, tgt_ids)
    print(f'JA : {ja}')
    print(f'EN : {translation}')
    print()